# ShiftGuard-X NB89 — Repaired Beijing/native codec replication and Q1 decision

This is a **standalone continuation** of NB88. The uploaded NB88 rerun had **0 Beijing stations loaded**, though four EPA streams completed. NB89 fixes nested UCI ZIP extraction, uses `from pcodec import standalone`, verifies exact float64 bytes, and compares real Gorilla/Chimp-family implementations with LZMA, Zstandard and SGX XOR+shuffle.

**Scientific constraints:** 25% chronological calibration/75% held-out; complete 22-byte file framing; no post-test selection; source hashes; PM2.5/PM10 primary and SO2/NO2 negative controls; a shuffled-order comparison. These are known coding methods, not an invented algorithm. Q1 novelty remains unproven.

Run all cells in a fresh Google Colab CPU runtime. Save `SGX_NB89_results.zip`. If a dependency or data source fails, the result is marked INCOMPLETE instead of fabricated.


In [ ]:

import sys,subprocess,importlib,os
INSTALL={}
for package,ver in [('zstandard','0.25.0'),('floatbungler','0.1.2'),('pcodec','1.0.3')]:
    try:
        __import__(package)
        INSTALL[package]='already installed'
    except ImportError:
        try:
            r=subprocess.run([sys.executable,'-m','pip','-q','install',f'{package}=={ver}'],capture_output=True,text=True,timeout=180)
            INSTALL[package]={'returncode':r.returncode,'error':r.stderr[-400:]}
        except Exception as e:INSTALL[package]={'error':repr(e)}
print(INSTALL)


In [ ]:

import io,csv,json,zipfile,struct,zlib,lzma,hashlib,urllib.request,time,statistics,platform,traceback
from pathlib import Path
from collections import Counter
import numpy as np,pandas as pd
try: import zstandard as zstd
except ImportError:zstd=None
try: from floatbungler import gorilla,chimp,chimp128,patas
except ImportError:gorilla=chimp=chimp128=patas=None
try: from pcodec import standalone as pcs
except ImportError:pcs=None
ROOT=Path('/content') if Path('/content').exists() else Path('/mnt/data')
OUT=ROOT/'SGX_NB89_outputs';OUT.mkdir(exist_ok=True)
def sha(b):return hashlib.sha256(b).hexdigest()
def raw(x):return np.asarray(x,dtype='<f8').copy().tobytes()
def dfwrite(name,rows):
    if not rows: rows=[{'status':'NO_ROWS'}]
    fields=list(dict.fromkeys(k for r in rows for k in r))
    with (OUT/name).open('w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields);w.writeheader();w.writerows(rows)
def jwrite(name,obj):(OUT/name).write_text(json.dumps(obj,indent=2,default=str))
CODECS={'gorilla':gorilla,'chimp':chimp,'chimp128':chimp128,'patas':patas}
METHODS=['zlib6','lzma1','zstd3','xor_shuffle_zstd3','pcodec','gorilla','chimp','chimp128','patas']
MID={m:i for i,m in enumerate(METHODS)}
HEADER=struct.Struct('<5sBQQ')
def xor(b):
    v=np.frombuffer(b,dtype='<u8');r=v.copy()
    if len(v)>1:r[1:]=v[1:]^v[:-1]
    return r.tobytes()
def unxor(b):return np.bitwise_xor.accumulate(np.frombuffer(b,dtype='<u8')).tobytes()
def shuffle(b):return np.frombuffer(b,dtype='u1').reshape(-1,8).T.copy().tobytes()
def unshuffle(b,n):return np.frombuffer(b,dtype='u1').reshape(8,n).T.copy().tobytes()
def available(m):
    if 'zstd' in m:return zstd is not None
    if m=='pcodec':return pcs is not None
    if m in CODECS:return CODECS[m] is not None
    return True
def encode(m,b):
    if len(b)%8:raise ValueError('float64 alignment')
    if m=='zlib6':p=zlib.compress(b,6)
    elif m=='lzma1':p=lzma.compress(b,preset=1)
    elif m=='zstd3':p=zstd.ZstdCompressor(level=3).compress(b)
    elif m=='xor_shuffle_zstd3':p=zstd.ZstdCompressor(level=3).compress(shuffle(xor(b)))
    elif m=='pcodec':p=bytes(pcs.simple_compress(np.frombuffer(b,dtype='<f8'),__import__('pcodec').ChunkConfig()))
    elif m in CODECS and CODECS[m] is not None:p=bytes(CODECS[m].encode(np.frombuffer(b,dtype='<f8').tolist()))
    else:raise ValueError('unavailable codec '+m)
    return HEADER.pack(b'SGX89',MID[m],len(b)//8,len(p))+p
def decode(blob):
    if len(blob)<HEADER.size:raise ValueError('short header')
    magic,i,n,pl=HEADER.unpack(blob[:HEADER.size])
    if magic!=b'SGX89' or i>=len(METHODS) or len(blob)!=HEADER.size+pl:raise ValueError('bad frame')
    m=METHODS[i];p=blob[HEADER.size:]
    if m=='zlib6':b=zlib.decompress(p)
    elif m=='lzma1':b=lzma.decompress(p)
    elif m=='zstd3':b=zstd.ZstdDecompressor().decompress(p)
    elif m=='xor_shuffle_zstd3':b=unxor(unshuffle(zstd.ZstdDecompressor().decompress(p),n))
    elif m=='pcodec':b=raw(pcs.simple_decompress(p))
    else:b=raw(CODECS[m].decode(p,n))
    if len(b)!=n*8:raise ValueError('reconstructed length mismatch')
    return b
def measure(m,b,reps=2):
    numbers=[]
    for _ in range(reps):
        t=time.perf_counter_ns();blob=encode(m,b);et=(time.perf_counter_ns()-t)/1e6
        t=time.perf_counter_ns();decoded=decode(blob);dt=(time.perf_counter_ns()-t)/1e6
        if decoded!=b:raise AssertionError('bit mismatch')
        numbers.append((len(blob),et,dt))
    return {'bytes':numbers[0][0],'encode_ms':statistics.median(x[1] for x in numbers),'decode_ms':statistics.median(x[2] for x in numbers)}
print('Methods available:',{m:available(m) for m in METHODS},'header',HEADER.size)


In [ ]:

# Verify exact bit patterns. Invalid pcodec or third-party methods are logged, not relabeled.
bits=np.array([0,0x8000000000000000,0x7ff0000000000000,0xfff0000000000000,0x7ff8000000000001,0x7ff8000000000a77,1,0xffffffffffffffff],dtype='<u8')
edge=[]
for m in METHODS:
    if not available(m):edge.append({'method':m,'status':'UNAVAILABLE'});continue
    try:
        b=bits.tobytes(); r=measure(m,b,1)
        edge.append({'method':m,'status':'PASS','bytes':r['bytes']})
    except Exception as e:edge.append({'method':m,'status':'INAPPLICABLE_OR_FAILED','error':repr(e)})
dfwrite('bitwise_edges.csv',edge)
print(edge)


In [ ]:

# Repaired recursive nested-ZIP ingestion. Source-provenance SHA-256 refers to the exact original archive.
URL='https://archive.ics.uci.edu/static/public/501/beijing+multi+site+air+quality+data.zip'
STATIONS=['Aotizhongxin','Changping','Dongsi','Guanyuan','Gucheng','Shunyi','Wanliu','Wanshouxigong']
CHANNELS=['PM2.5','PM10','SO2','NO2']
provenance=[];issues=[];streams=[]
def download():
    req=urllib.request.Request(URL,headers={'User-Agent':'SGX-NB89-reproducible-research'})
    with urllib.request.urlopen(req,timeout=100) as r:return r.read()
def csv_files(archive,depth=0,prefix=''):
    if depth>3:raise ValueError('nested ZIP depth exceeded')
    with zipfile.ZipFile(io.BytesIO(archive)) as z:
        for item in z.namelist():
            if item.lower().endswith('.zip'):
                yield from csv_files(z.read(item),depth+1,prefix+item+'::')
            elif item.lower().endswith('.csv'):
                yield prefix+item,z.read(item)
try:
    archive=download()
    found=list(csv_files(archive))
    provenance.append({'dataset':'Beijing UCI 501','url':URL,'sha256':sha(archive),'bytes':len(archive),'csv_members':len(found)})
    for station in STATIONS:
        match=[(path,b) for path,b in found if 'PRSA_Data_'+station+'_' in path]
        if len(match)!=1:issues.append({'station':station,'error':'CSV count '+str(len(match))});continue
        path,b=match[0];d=pd.read_csv(io.BytesIO(b))
        d=d.sort_values(['year','month','day','hour'],kind='stable')
        for channel in CHANNELS:
            vals=pd.to_numeric(d[channel],errors='coerce').to_numpy(dtype='<f8')[:32000]
            if len(vals)<1024:issues.append({'station':station,'error':'too few observations'});continue
            streams.append({'station':station,'channel':channel,'name':station+'__'+channel,'values':vals,'sha256_full':sha(raw(vals)),'csv_member':path})
except Exception as exc:
    issues.append({'station':'ALL','error':repr(exc)})
dfwrite('source_provenance.csv',provenance)
dfwrite('source_errors.csv',issues)
print('Stations loaded:',len(set(s['station'] for s in streams)),'streams',len(streams),'errors',issues[:3])


In [ ]:

# Every policy choice is made from calibration. The held-out comparison uses the SAME bytes.
results=[];traces=[];failures=[];controls=[]
BASE=['zlib6','lzma1','zstd3'];NATIVE=['pcodec','gorilla','chimp','chimp128','patas']
for s in streams:
    v=s['values'];cut=len(v)//4;cal=raw(v[:cut]);test=raw(v[cut:]);scores={}
    for m in METHODS:
        if not available(m):continue
        try:scores[m]=measure(m,cal,1)
        except Exception as e:failures.append({'stream':s['name'],'stage':'calibration','method':m,'reason':repr(e)})
    if not any(m in scores for m in BASE):failures.append({'stream':s['name'],'stage':'selection','method':'ALL','reason':'No baseline'});continue
    select=lambda options:min((m for m in options if m in scores),key=lambda m:scores[m]['bytes'],default=None)
    picked={'generic':select(BASE),'native':select(NATIVE),'sgx':select(['xor_shuffle_zstd3'])}
    held={}
    for m in set(x for x in picked.values() if x):
        try:held[m]=measure(m,test,2)
        except Exception as e:failures.append({'stream':s['name'],'stage':'heldout','method':m,'reason':repr(e)})
    rec={'station':s['station'],'channel':s['channel'],'stream':s['name'],'n_cal':cut,'n_test':len(v)-cut,'heldout_sha256':sha(test),'primary':s['channel'] in ('PM2.5','PM10')}
    for k,m in picked.items():
        rec[k+'_method']=m
        rec[k+'_bytes']=held[m]['bytes'] if m in held else None
        if m in held:rec[k+'_encode_ms']=held[m]['encode_ms']
    if rec.get('generic_bytes') and rec.get('native_bytes'):
        rec['native_gain_vs_generic_pct']=100*(rec['generic_bytes']-rec['native_bytes'])/rec['generic_bytes']
    if rec.get('generic_bytes') and rec.get('sgx_bytes'):
        rec['sgx_gain_vs_generic_pct']=100*(rec['generic_bytes']-rec['sgx_bytes'])/rec['generic_bytes']
    results.append(rec)
    if s['channel']=='PM2.5':
        perm=np.random.default_rng(8902).permutation(len(v)-cut)
        scrambled=raw(v[cut:][perm])
        check={'stream':s['name'],'seed':8902}
        for k,m in picked.items():
            if not m:continue
            try:check[k+'_shuffled_bytes']=measure(m,scrambled,1)['bytes']
            except Exception as e:check[k+'_error']=repr(e)
        controls.append(check)
dfwrite('heldout_frozen_decisions.csv',results)
dfwrite('codec_errors.csv',failures)
dfwrite('shuffled_negative_controls.csv',controls)
groups=[]
for (channel,primary),v in __import__('itertools').groupby(sorted(results,key=lambda x:(x['channel'],x['primary'])),key=lambda x:(x['channel'],x['primary'])):
    rr=list(v);eligible=[r for r in rr if r.get('generic_bytes') and r.get('native_bytes') and r.get('sgx_bytes')]
    if eligible:
        gb=sum(r['generic_bytes'] for r in eligible);nb=sum(r['native_bytes'] for r in eligible);sb=sum(r['sgx_bytes'] for r in eligible)
        groups.append({'channel':channel,'primary':primary,'n':len(eligible),'generic_bytes':gb,'native_bytes':nb,'sgx_bytes':sb,'sgx_saving_vs_generic_pct':100*(gb-sb)/gb,'sgx_saving_vs_native_pct':100*(nb-sb)/nb})
dfwrite('channel_summary.csv',groups)
complete=len(results)==len(STATIONS)*len(CHANNELS) and len(issues)==0 and all(r.get('native_bytes') is not None and r.get('sgx_bytes') is not None for r in results)
primary=[r for r in results if r['primary'] and r.get('native_bytes') and r.get('sgx_bytes') and r.get('generic_bytes')]
win=sum(r['sgx_bytes']<min(r['native_bytes'],r['generic_bytes']) for r in primary)
decision={'status':'COMPLETE' if complete else 'INCOMPLETE','loaded_stations':len(set(r['station'] for r in results)),'streams':len(results),
 'primary_complete_streams':len(primary),'SGX_wins_vs_both_native_and_generic':win,
 'research_decision':'DO_NOT_CLAIM_NOVELTY_OR_Q1_READY','next_step':'If no meaningful gain against BOTH native and generic baselines, retire this temporal-XOR direction. Any positive finding requires independent validation and a genuinely new mechanism.',
 'note':'NB89 uses the official UCI Beijing source. EPA findings from NB88 remain independent, previously collected evidence, not rerun here.',
 'source_errors':issues,'codec_errors':len(failures)}
jwrite('decision.json',decision)
jwrite('environment.json',{'python':sys.version,'numpy':np.__version__,'versions':INSTALL})
print(json.dumps(decision,indent=2))
path=ROOT/'SGX_NB89_results.zip'
with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
    for file in OUT.iterdir():
        if file.is_file():z.write(file,file.name)
print('Result archive:',path)
try:
    from google.colab import files;files.download(str(path))
except ImportError:pass
